# NB_bishop_ch18_figures
## Key Figures for Bishop Chapter 18 -- Normalizing Flows

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/NEURAL_BIZ/blob/master/Bishop_Ch_18/NB_bishop_ch18_figures.ipynb)

Generate publication-quality figures for normalizing flows: change of variables, planar flows, coupling layers, RealNVP, and flow composition.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib

matplotlib.rcParams['figure.facecolor'] = 'none'
matplotlib.rcParams['axes.facecolor'] = 'none'
matplotlib.rcParams['savefig.facecolor'] = 'none'
matplotlib.rcParams['savefig.transparent'] = True
matplotlib.rcParams['axes.grid'] = False
matplotlib.rcParams['legend.loc'] = 'upper center'
matplotlib.rcParams['legend.framealpha'] = 0.0

COLORS = {'blue': '#1A3A6E', 'red': '#CD0000', 'green': '#2E7D32', 'amber': '#B5853F'}

def save_fig(fig, name, chart_dir='../../charts'):
    for ax in fig.get_axes():
        ax.grid(False)
        legend = ax.get_legend()
        if legend:
            legend.set_bbox_to_anchor((0.5, -0.15))
            legend.set_loc('upper center')
            legend._ncols = min(len(legend.get_texts()), 4)
    fig.savefig(f'{chart_dir}/{name}.pdf', bbox_inches='tight', transparent=True)
    fig.savefig(f'{chart_dir}/{name}.png', bbox_inches='tight', transparent=True, dpi=150)
    print(f'Saved {chart_dir}/{name}.pdf and .png')

np.random.seed(42)

## Figure 18.1 -- Change of Variables

1D change of variables: a source density $p_z(z)$ (standard Gaussian) is pushed through an invertible transformation $x = f(z)$, producing a target density $p_x(x) = p_z(f^{-1}(x)) \left|\frac{df^{-1}}{dx}\right|$.

In [ ]:
from scipy.stats import norm

z = np.linspace(-4, 4, 500)
pz = norm.pdf(z)

# Invertible transformation: x = f(z) = z + 0.6*sin(2z)  (smooth, invertible for this range)
def f(z):
    return z + 0.6 * np.sin(2 * z)

def f_deriv(z):
    return 1 + 1.2 * np.cos(2 * z)

x_vals = f(z)
# p_x(x) = p_z(z) / |f'(z)|
px = pz / np.abs(f_deriv(z))

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

# Panel 1: Source density
ax = axes[0]
ax.fill_between(z, pz, alpha=0.15, color=COLORS['blue'])
ax.plot(z, pz, color=COLORS['blue'], linewidth=2.5)
ax.set_xlabel('$z$', fontsize=12)
ax.set_ylabel('$p_z(z)$', fontsize=12)
ax.set_title('Source Density (Gaussian)', fontweight='bold', fontsize=12)
ax.set_xlim(-4, 4)

# Panel 2: Transformation f
ax = axes[1]
ax.plot(z, x_vals, color=COLORS['amber'], linewidth=2.5)
ax.plot(z, z, color='grey', linewidth=1, linestyle='--', alpha=0.5, label='Identity')
ax.set_xlabel('$z$', fontsize=12)
ax.set_ylabel('$x = f(z)$', fontsize=12)
ax.set_title('Invertible Transformation', fontweight='bold', fontsize=12)
ax.legend(fontsize=10)
ax.set_xlim(-4, 4)

# Panel 3: Target density
ax = axes[2]
ax.fill_between(x_vals, px, alpha=0.15, color=COLORS['red'])
# Sort for clean plotting
sort_idx = np.argsort(x_vals)
ax.plot(x_vals[sort_idx], px[sort_idx], color=COLORS['red'], linewidth=2.5)
ax.set_xlabel('$x$', fontsize=12)
ax.set_ylabel('$p_x(x)$', fontsize=12)
ax.set_title('Target Density (Transformed)', fontweight='bold', fontsize=12)
ax.set_xlim(-5, 5)

# Add formula annotation (use simple LaTeX compatible with matplotlib)
fig.text(0.5, -0.06,
         r'$p_x(x) = p_z(f^{-1}(x)) \cdot \left|\det \frac{\partial f^{-1}}{\partial x}\right|$',
         ha='center', fontsize=14, color=COLORS['blue'])

fig.suptitle('Change of Variables Formula', fontweight='bold', fontsize=14, y=1.03)
fig.tight_layout()
save_fig(fig, 'fig18_1_change_of_variables')
plt.show()

## Figure 18.3 -- Planar Flow Transformation

A standard 2D Gaussian is warped by successive planar flow layers $f(\mathbf{z}) = \mathbf{z} + \mathbf{u}\,h(\mathbf{w}^\top \mathbf{z} + b)$, showing progressive deformation from simple to complex.

In [ ]:
np.random.seed(42)

# Planar flow: f(z) = z + u * tanh(w^T z + b)
# Define several layers with different parameters
flow_params = [
    {'w': np.array([1.0, 0.5]), 'u': np.array([0.8, -0.3]), 'b': -0.5},
    {'w': np.array([-0.5, 1.2]), 'u': np.array([0.3, 0.9]), 'b': 0.3},
    {'w': np.array([0.7, -0.8]), 'u': np.array([-0.6, 0.5]), 'b': 0.1},
    {'w': np.array([1.1, 0.3]), 'u': np.array([0.4, -0.7]), 'b': -0.2},
    {'w': np.array([-0.3, 1.0]), 'u': np.array([0.7, 0.6]), 'b': 0.4},
]

def planar_flow(z, w, u, b):
    return z + np.outer(np.tanh(z @ w + b), u)

# Sample from base distribution
N = 8000
z0 = np.random.randn(N, 2)

# Apply successive layers
stages = [z0.copy()]
z = z0.copy()
for p in flow_params:
    z = planar_flow(z, p['w'], p['u'], p['b'])
    stages.append(z.copy())

titles = ['Base $z_0$', 'After 1 layer', 'After 2 layers',
          'After 3 layers', 'After 4 layers', 'After 5 layers']
colors_stage = [COLORS['blue'], COLORS['blue'], COLORS['amber'],
                COLORS['amber'], COLORS['red'], COLORS['red']]

fig, axes = plt.subplots(2, 3, figsize=(14, 9))
for idx, (ax, pts, title, c) in enumerate(zip(axes.flat, stages, titles, colors_stage)):
    ax.scatter(pts[:, 0], pts[:, 1], s=1.0, alpha=0.25, color=c, rasterized=True)
    ax.set_title(title, fontweight='bold', fontsize=12)
    ax.set_xlim(-4, 4)
    ax.set_ylim(-4, 4)
    ax.set_aspect('equal')
    ax.set_xlabel('$z_1$', fontsize=10)
    ax.set_ylabel('$z_2$', fontsize=10)

fig.suptitle('Planar Flow: Progressive Warping of a Gaussian',
             fontweight='bold', fontsize=14, y=1.02)
fig.tight_layout()
save_fig(fig, 'fig18_3_planar_flow')
plt.show()

## Figure 18.4 -- Coupling Layer Diagram

An affine coupling layer splits the input into two halves: $\mathbf{z}_{1:d}$ passes through unchanged, while $\mathbf{z}_{d+1:D}$ is transformed using scale and translation functions conditioned on $\mathbf{z}_{1:d}$.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))
ax.set_xlim(-1, 17)
ax.set_ylim(-1, 9)
ax.axis('off')

def draw_box(ax, x, y, w, h, label, fc, ec, fs=11):
    box = mpatches.FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.15',
                                   facecolor=fc, edgecolor=ec, linewidth=2)
    ax.add_patch(box)
    ax.text(x + w/2, y + h/2, label, ha='center', va='center',
            fontsize=fs, fontweight='bold', color=ec)

def arrow(ax, x1, y1, x2, y2, color='black', lw=2, style='->', ls='-'):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle=style, color=color, lw=lw, linestyle=ls))

# Input
draw_box(ax, 0, 5.5, 2.5, 1.2, 'Input $\\mathbf{z}$', '#E8E8E8', '#333333', 12)

# Split
ax.text(4.0, 6.1, 'Split', fontsize=11, ha='center', va='center',
        fontweight='bold', color='#555555')
arrow(ax, 2.5, 6.1, 5.0, 7.2, color='#555555')
arrow(ax, 2.5, 6.1, 5.0, 4.8, color='#555555')

# Top path: z_{1:d} passes through (identity)
draw_box(ax, 5.0, 6.8, 2.8, 1.0, '$\\mathbf{z}_{1:d}$\n(unchanged)', '#D6E4F0', COLORS['blue'], 11)
arrow(ax, 7.8, 7.3, 12.5, 7.3, color=COLORS['blue'])

# Bottom path: z_{d+1:D} is transformed
draw_box(ax, 5.0, 4.0, 2.8, 1.0, '$\\mathbf{z}_{d+1:D}$', '#FCE4EC', COLORS['red'], 11)

# Neural network blocks s() and t()
draw_box(ax, 5.3, 1.5, 2.2, 1.0, '$s(\\mathbf{z}_{1:d})$', '#FFF3E0', COLORS['amber'], 10)
draw_box(ax, 8.3, 1.5, 2.2, 1.0, '$t(\\mathbf{z}_{1:d})$', '#E8F5E9', COLORS['green'], 10)

# Arrow from z_{1:d} down to s and t
arrow(ax, 6.4, 6.8, 6.4, 2.5, color=COLORS['blue'], lw=1.5, ls='--')
arrow(ax, 6.4, 6.8, 9.4, 2.5, color=COLORS['blue'], lw=1.5, ls='--')

# Multiply node
circle_x, circle_y = 9.0, 4.5
circle = plt.Circle((circle_x, circle_y), 0.4, fill=False, edgecolor=COLORS['amber'], linewidth=2)
ax.add_patch(circle)
ax.text(circle_x, circle_y, '$\\odot$', ha='center', va='center', fontsize=16, color=COLORS['amber'])

# Arrow z_{d+1:D} -> multiply
arrow(ax, 7.8, 4.5, 8.6, 4.5, color=COLORS['red'])

# Arrow s -> multiply
arrow(ax, 6.4, 2.5, circle_x - 0.1, circle_y - 0.4, color=COLORS['amber'], lw=1.5)

# Add node
plus_x, plus_y = 11.0, 4.5
circle2 = plt.Circle((plus_x, plus_y), 0.4, fill=False, edgecolor=COLORS['green'], linewidth=2)
ax.add_patch(circle2)
ax.text(plus_x, plus_y, '$+$', ha='center', va='center', fontsize=16,
        fontweight='bold', color=COLORS['green'])

# Arrow multiply -> add
arrow(ax, circle_x + 0.4, circle_y, plus_x - 0.4, plus_y, color='#333333')

# Arrow t -> add
arrow(ax, 9.4, 2.5, plus_x - 0.1, plus_y - 0.4, color=COLORS['green'], lw=1.5)

# Output bottom path
draw_box(ax, 12.5, 4.0, 3.0, 1.0,
         '$\\mathbf{y}_{d+1:D} = s \\odot z + t$', '#FCE4EC', COLORS['red'], 10)
arrow(ax, plus_x + 0.4, plus_y, 12.5, 4.5, color=COLORS['red'])

# Output top path
draw_box(ax, 12.5, 6.8, 3.0, 1.0, '$\\mathbf{y}_{1:d} = \\mathbf{z}_{1:d}$', '#D6E4F0', COLORS['blue'], 11)

# Concat
ax.text(16.2, 6.1, 'Concat', fontsize=11, ha='center', va='center',
        fontweight='bold', color='#555555')
arrow(ax, 15.5, 7.2, 16.2, 6.5, color='#555555')
arrow(ax, 15.5, 4.8, 16.2, 5.7, color='#555555')

# Output
draw_box(ax, 16.5, 5.5, 0.3, 1.2, '', '#E8E8E8', '#333333', 12)
ax.text(16.65, 6.1, '$\\mathbf{y}$', fontsize=13, fontweight='bold',
        ha='center', va='center', color='#333333')

# Title and note
ax.set_title('Affine Coupling Layer', fontweight='bold', fontsize=15, pad=15)
ax.text(8.5, -0.3,
        'Jacobian is triangular $\\Rightarrow$ $\\det = \\prod s_i$ (cheap to compute)',
        ha='center', fontsize=12, color=COLORS['blue'], fontstyle='italic')

fig.tight_layout()
save_fig(fig, 'fig18_4_coupling_layer')
plt.show()

## Figure 18.5 -- Flow Composition

Chain of invertible transformations $\mathbf{z}_0 \xrightarrow{f_1} \mathbf{z}_1 \xrightarrow{f_2} \mathbf{z}_2 \xrightarrow{f_3} \cdots \xrightarrow{f_K} \mathbf{x}$, showing how a simple base distribution is progressively transformed into a complex target.

In [ ]:
from scipy.stats import norm as norm_dist
from matplotlib.patches import FancyArrowPatch

fig, ax = plt.subplots(figsize=(16, 5))
ax.set_xlim(-1, 21)
ax.set_ylim(-2, 5)
ax.axis('off')

# Positions for distribution bubbles
positions = [1, 5, 9, 13, 17]
labels_top = ['$\\mathbf{z}_0$', '$\\mathbf{z}_1$', '$\\mathbf{z}_2$', '$\\mathbf{z}_3$', '$\\mathbf{x}$']
labels_bot = ['$p_0(\\mathbf{z}_0)$', '', '', '', '$p_x(\\mathbf{x})$']
flow_labels = ['$f_1$', '$f_2$', '$f_3$', '$f_K$']

# Draw each distribution as a small density inset
np.random.seed(42)
# Generate progressively more complex 1D densities
t_vals = np.linspace(-3, 3, 200)
densities = [
    norm_dist.pdf(t_vals),  # z0: simple Gaussian
    0.7 * norm_dist.pdf(t_vals, -0.5, 0.8) + 0.3 * norm_dist.pdf(t_vals, 1.5, 0.6),  # z1
    0.4 * norm_dist.pdf(t_vals, -1.2, 0.5) + 0.35 * norm_dist.pdf(t_vals, 0.5, 0.7) + \
        0.25 * norm_dist.pdf(t_vals, 2.0, 0.4),  # z2
    0.3 * norm_dist.pdf(t_vals, -1.5, 0.4) + 0.25 * norm_dist.pdf(t_vals, -0.2, 0.5) + \
        0.25 * norm_dist.pdf(t_vals, 1.0, 0.3) + 0.2 * norm_dist.pdf(t_vals, 2.2, 0.5),  # z3
    0.2 * norm_dist.pdf(t_vals, -2.0, 0.35) + 0.25 * norm_dist.pdf(t_vals, -0.5, 0.4) + \
        0.3 * norm_dist.pdf(t_vals, 1.0, 0.3) + 0.25 * norm_dist.pdf(t_vals, 2.3, 0.45),  # x
]
colors_flow = [COLORS['blue'], COLORS['blue'], COLORS['amber'], COLORS['amber'], COLORS['red']]

for i, (xc, lbl, lbl_b, dens, c) in enumerate(zip(positions, labels_top, labels_bot,
                                                      densities, colors_flow)):
    # Draw mini density plot
    inset_w, inset_h = 2.5, 2.0
    x_left, y_bot = xc - inset_w / 2, 0.5
    # Normalize density for plotting
    dens_norm = dens / dens.max() * inset_h * 0.85
    x_plot = x_left + (t_vals - t_vals.min()) / (t_vals.max() - t_vals.min()) * inset_w
    y_plot = y_bot + dens_norm

    ax.fill_between(x_plot, y_bot, y_plot, alpha=0.15, color=c)
    ax.plot(x_plot, y_plot, color=c, linewidth=2)

    # Border
    rect = mpatches.FancyBboxPatch((x_left - 0.15, y_bot - 0.15), inset_w + 0.3, inset_h + 0.15,
                                    boxstyle='round,pad=0.1', facecolor='none',
                                    edgecolor=c, linewidth=1.5, linestyle='--', alpha=0.5)
    ax.add_patch(rect)

    # Labels
    ax.text(xc, inset_h + y_bot + 0.5, lbl, ha='center', va='center',
            fontsize=14, fontweight='bold', color=c)
    if lbl_b:
        ax.text(xc, y_bot - 0.5, lbl_b, ha='center', va='center',
                fontsize=11, color=c, fontstyle='italic')

# Draw arrows between distributions
for i in range(len(positions) - 1):
    x_start = positions[i] + 1.4
    x_end = positions[i + 1] - 1.4

    if i == 2:  # dots between z2 and z3 to indicate "..."
        ax.text((positions[i] + positions[i + 1]) / 2, 1.5, '$\\cdots$',
                ha='center', va='center', fontsize=20, color='#555555')
    else:
        ax.annotate('', xy=(x_end, 1.5), xytext=(x_start, 1.5),
                    arrowprops=dict(arrowstyle='->', color='#333333', lw=2.5))

    ax.text((positions[i] + positions[i + 1]) / 2, 3.8, flow_labels[i],
            ha='center', va='center', fontsize=13, fontweight='bold',
            color=COLORS['amber'],
            bbox=dict(boxstyle='round,pad=0.3', facecolor='#FFF3E0',
                      edgecolor=COLORS['amber'], linewidth=1.5))
    # Curved arrow from label to main arrow
    ax.annotate('', xy=((positions[i] + positions[i + 1]) / 2, 1.8),
                xytext=((positions[i] + positions[i + 1]) / 2, 3.4),
                arrowprops=dict(arrowstyle='->', color=COLORS['amber'],
                                lw=1, linestyle='--', connectionstyle='arc3,rad=0'))

# Log-likelihood formula at bottom
ax.text(9, -1.5,
        r'$\ln p_x(\mathbf{x}) = \ln p_0(\mathbf{z}_0) + \sum_{k=1}^{K} \ln\left|\det \frac{\partial f_k}{\partial \mathbf{z}_{k-1}}\right|$',
        ha='center', fontsize=14, color=COLORS['blue'],
        bbox=dict(boxstyle='round,pad=0.4', facecolor='white',
                  edgecolor=COLORS['blue'], linewidth=1.5, alpha=0.8))

ax.set_title('Flow Composition: Chain of Invertible Transformations',
             fontweight='bold', fontsize=14, pad=15)

fig.tight_layout()
save_fig(fig, 'fig18_5_flow_composition')
plt.show()

## Figure 18.6 -- RealNVP Alternating Masks

RealNVP uses alternating binary masks in successive coupling layers so that every dimension is transformed. Shows checkerboard and channel-wise masking patterns.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(14, 7))

grid_size = 8

# Top row: Checkerboard masking (alternating layers)
for col in range(4):
    ax = axes[0, col]
    mask = np.zeros((grid_size, grid_size))
    for i in range(grid_size):
        for j in range(grid_size):
            if (i + j) % 2 == (col % 2):
                mask[i, j] = 1

    # Color: masked (unchanged) = blue, transformed = red
    rgb = np.zeros((grid_size, grid_size, 3))
    blue_rgb = np.array([int(COLORS['blue'][i:i+2], 16) / 255 for i in (1, 3, 5)])
    red_rgb = np.array([int(COLORS['red'][i:i+2], 16) / 255 for i in (1, 3, 5)])

    for i in range(grid_size):
        for j in range(grid_size):
            if mask[i, j] == 1:
                rgb[i, j] = blue_rgb  # unchanged (masked)
            else:
                rgb[i, j] = red_rgb   # transformed

    ax.imshow(rgb, interpolation='nearest')
    # Grid lines
    for k in range(grid_size + 1):
        ax.axhline(k - 0.5, color='white', linewidth=1)
        ax.axvline(k - 0.5, color='white', linewidth=1)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(f'Layer {col + 1}', fontweight='bold', fontsize=11)

axes[0, 0].set_ylabel('Checkerboard\nMasking', fontsize=11, fontweight='bold',
                       rotation=0, labelpad=80, va='center')

# Bottom row: Channel-wise (half-split) masking
for col in range(4):
    ax = axes[1, col]
    mask = np.zeros((grid_size, grid_size))

    if col % 2 == 0:
        # Top half unchanged, bottom half transformed
        mask[:grid_size // 2, :] = 1
    else:
        # Left half unchanged, right half transformed
        mask[:, :grid_size // 2] = 1

    rgb = np.zeros((grid_size, grid_size, 3))
    for i in range(grid_size):
        for j in range(grid_size):
            if mask[i, j] == 1:
                rgb[i, j] = blue_rgb
            else:
                rgb[i, j] = red_rgb

    ax.imshow(rgb, interpolation='nearest')
    for k in range(grid_size + 1):
        ax.axhline(k - 0.5, color='white', linewidth=1)
        ax.axvline(k - 0.5, color='white', linewidth=1)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(f'Layer {col + 1}', fontweight='bold', fontsize=11)

axes[1, 0].set_ylabel('Channel-wise\nMasking', fontsize=11, fontweight='bold',
                       rotation=0, labelpad=80, va='center')

# Legend
from matplotlib.lines import Line2D
legend_elements = [
    mpatches.Patch(facecolor=COLORS['blue'], label='Unchanged (identity)'),
    mpatches.Patch(facecolor=COLORS['red'], label='Transformed (affine coupling)'),
]
fig.legend(handles=legend_elements, loc='upper center', bbox_to_anchor=(0.5, -0.02),
           ncol=2, framealpha=0.0, fontsize=11)

fig.suptitle('RealNVP: Alternating Masking Patterns Across Coupling Layers',
             fontweight='bold', fontsize=14, y=1.02)
fig.tight_layout()
save_fig(fig, 'fig18_6_realNVP')
plt.show()

## Summary

- **fig18_1**: Change of variables -- source Gaussian, invertible transformation, and Jacobian-corrected target density
- **fig18_3**: Planar flow -- progressive warping of a 2D Gaussian through 5 planar flow layers
- **fig18_4**: Coupling layer -- architectural diagram of an affine coupling layer with scale/translation networks
- **fig18_5**: Flow composition -- chain of invertible transformations with increasingly complex distributions
- **fig18_6**: RealNVP -- alternating checkerboard and channel-wise masking patterns across coupling layers